# Jester GW170817 Google Colab example

This example shows how to run the jester inference on a cloud GPU offered by Google inside Google Colab, for more expensive inferences such as analyzing GW170817.

**How to connect to a GPU on Google Colab?**

1. Click on the dropdown menu in the top-right next to "Connect", and click "Change runtime type"
2. Choose the T4 GPU. This one is offered for free from Google Colab and is good enough for the purposes of this demonstration. We recommend, however, to run jester on higher-end GPUs if available to you.

If you now hover over the "RAM/Disk" icon, you should now see something like "Connected to Python 3 Google Compute Engine backend (GPU)"

We start by installing `jester` with a uv virtual environment, making sure we install support for GPUs

## Installing jester on Google Colab

In [10]:
!rm -rf jester/ .venv/
!git clone https://github.com/nuclear-multimessenger-astronomy/jester
%cd jester
!uv venv --python=3.12
!source .venv/bin/activate && uv pip install -e ".[cuda12]"

Cloning into 'jester'...
remote: Enumerating objects: 7195, done.
remote: Counting objects: 100% (1801/1801), done.
remote: Compressing objects: 100% (744/744), done.
remote: Total 7195 (delta 1130), reused 1111 (delta 1055), pack-reused 5394 (from 3)
Receiving objects: 100% (7195/7195), 62.06 MiB | 19.68 MiB/s, done.
Resolving deltas: 100% (4623/4623), done.
/content/my_shared_experiment/jester
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Using Python 3.12.13 environment at: /usr
Resolved 117 packages in 986ms
Prepared 1 package in 368ms
Uninstalled 1 package in 0.48ms
Installed 1 package in 1ms
 ~ jestertov==0.2.0 (from file:///content/my_shared_experiment/jester)


## Setting up the inference

Now, we will navigate to an example running the SMC inference on GW170817. This is too expensive to run on most laptop CPUs, but becomes manageable once we run it on the GPU.

**NOTE**: The standard hyperparameters for the GW170817 inference are too costly for the standard GPU that one can use for free on Google Colab.

We will therefore navigate to the GW170817 example directory, and copy the jester `config.yaml` file and the `prior.prior` file to local copies. Then, we will edit these to

1. Reduce the number of particles in SMC
2. Reduce the number of masses on which the GW170817 EOS likelihood is evaluated

In order to use more advanced settings or sample more complicated likelihood surfaces (such as a collection of GW likelihoods from several BNSs), it is advised to try and run on higher-end GPUs, either through buying compute units in Google Colab or looking for access to a computing cluster with GPUs available.

In [11]:
import shutil
import os

# 1. Define paths
root_jester = "/content/jester"
example_dir = os.path.join(root_jester, "examples/inference/smc_random_walk/GW170817")
experiment_dir = "/content/my_shared_experiment"

# 2. Create the experiment folder and copy files
os.makedirs(experiment_dir, exist_ok=True)
shutil.copy(
    os.path.join(example_dir, "config.yaml"),
    os.path.join(experiment_dir, "config.yaml"),
)
shutil.copy(
    os.path.join(example_dir, "prior.prior"),
    os.path.join(experiment_dir, "prior.prior"),
)

# 3. Modify the config file using string replacement
config_path = os.path.join(experiment_dir, "config.yaml")

with open(config_path, "r") as f:
    content = f.read()

# Update SMC sampler settings
content = content.replace("n_particles: 5000", "n_particles: 2000")
content = content.replace("n_eos_samples: 5000", "n_eos_samples: 2000")

# Update GW likelihood settings
content = content.replace("N_masses_evaluation: 2000", "N_masses_evaluation: 200")
content = content.replace("N_masses_batch_size: 1000", "N_masses_batch_size: 200")

# Write back to file
with open(config_path, "w") as f:
    f.write(content)

print(f"Files prepared and modified in: {experiment_dir}")

Files prepared and modified in: /content/my_shared_experiment


In [12]:
# 1. Navigate to the example directory
%cd /content/my_shared_experiment

# 2. Display the configuration and prior files
print("--- Contents of config.yaml ---")
!cat config.yaml

print("\n--- Contents of prior.prior ---")
!cat prior.prior

/content/my_shared_experiment
--- Contents of config.yaml ---
seed: 43
dry_run: false
validate_only: false
eos:
  type: metamodel_cse
  ndat_metamodel: 100
  nmax_nsat: 25.0
  nb_CSE: 8
  crust_name: DH
tov:
  min_nsat_TOV: 0.75
  ndat_TOV: 100
  nb_masses: 100
  type: gr
prior:
  specification_file: prior.prior
likelihoods:
- type: constraints_eos
  enabled: true
- type: gw
  enabled: true
  events:
  - name: GW170817
  N_masses_evaluation: 200
  N_masses_batch_size: 200
  seed: 42
- type: radio
  enabled: true
  pulsars:
  - name: J1614
    mass_mean: 1.94
    mass_std: 0.06
  - name: J0740
    mass_mean: 2.08
    mass_std: 0.14
sampler:
  type: smc-rw
  n_particles: 2000
  n_mcmc_steps: 10
  target_ess: 0.9
  random_walk_sigma: 0.05
  n_eos_samples: 2000
  output_dir: ./outdir/
postprocessing:
  enabled: true
  make_cornerplot: true
  make_massradius: true
  make_pressuredensity: true

--- Contents of prior.prior ---
E_sat = UniformPrior(-16.1, -15.9, parameter_names=["E_sat"])
K_sa

## Launching the inference

Now, run the example with the command-line-interface: `run_jester_inference config.yaml` (making sure we activate the venv first)

In [13]:
!run_jester_inference config.yaml

[INFO] jester: Loading configuration from config.yaml
[INFO] jester: JAX devices: [CudaDevice(id=0)]
[INFO] jester: Setting up prior...
[INFO] jester: Prior has 27 dimensions
[INFO] jester: Prior parameter names: ['E_sat', 'K_sat', 'Q_sat', 'Z_sat', 'E_sym', 'L_sym', 'K_sym', 'Q_sym', 'Z_sym', 'nbreak', 'n_CSE_0_u', 'cs2_CSE_0', 'n_CSE_1_u', 'cs2_CSE_1', 'n_CSE_2_u', 'cs2_CSE_2', 'n_CSE_3_u', 'cs2_CSE_3', 'n_CSE_4_u', 'cs2_CSE_4', 'n_CSE_5_u', 'cs2_CSE_5', 'n_CSE_6_u', 'cs2_CSE_6', 'n_CSE_7_u', 'cs2_CSE_7', 'cs2_CSE_8']
[INFO] jester:   [0] E_sat: Uniform(-16.1, -15.9)
[INFO] jester:   [1] K_sat: Uniform(150.0, 300.0)
[INFO] jester:   [2] Q_sat: Uniform(-500.0, 1100.0)
[INFO] jester:   [3] Z_sat: Uniform(-2500.0, 1500.0)
[INFO] jester:   [4] E_sym: Uniform(28.0, 45.0)
[INFO] jester:   [5] L_sym: Uniform(10.0, 200.0)
[INFO] jester:   [6] K_sym: Uniform(-400.0, 200.0)
[INFO] jester:   [7] Q_sym: Uniform(-1000.0, 1500.0)
[INFO] jester:   [8] Z_sym: Uniform(-2000.0, 1500.0)
[INFO] jester: 

The inference ran successfully and completed sampling in around 15 minutes, with a few extra minutes required for setting up the inference and performing the postprocessing.

The output files can be found in the left sidebar, by clicking on the folder icon, at /content/my_shared_experiment/outdir

For reference, on an NVIDIA H100 GPU, the inference with the original settings (prior to the modifications performed within this notebook) finishes sampling in around 5 minutes.